# Afternoon class 30/08 — Worksheet 01 SOLUTIONS: Series   (L01)

Every cell below was executed in the lab image (pandas 3.0.5, Python 3.13.15)
and the quoted output is what it actually printed — including the error in Q10.

Question 3 is the one to re-read. The slide rounds a number that Python does
not round, and the difference is not cosmetic.

Run this cell once to set up the data. Then work down the sheet.

In [ ]:
# Worksheet 01 — Series. Run this once.
import pandas as pd

# The three students from the lecture slides.
ages = pd.Series([22, 24, 21], index=["Abdullah", "Sara", "Ahmed"])

# A plain Series with no index given, so Pandas supplies one.
plain = pd.Series([10, 20, 30])

# Marks, used later for filtering.
marks = pd.Series([85, 90, 75], index=["S1", "S2", "S3"])

print(ages)
print()
print("pandas version:", pd.__version__)

PART A — values and labels

### Question 1

Values and labels are stored separately. -> `values: [22 24 21]` and `index: Index(['Abdullah', 'Sara', 'Ahmed'], dtype='str')`.

`.values` gives you a bare NumPy array — no labels on it at all. The
labels live in a separate `Index` object. A Series is those two things
held together, which is precisely what a list cannot do.

Note the `dtype='str'`. The lecture slide shows `dtype="object"` for this
exact expression. Both are 'correct' answers for some version of Pandas —
`object` was right for years — but this lab runs pandas 3.0.5, where the
default string dtype changed. If a tutorial you find online prints
`object` here, it is older than your Pandas, not wrong.

In [ ]:
print(ages)
print()
print("values:", ages.values)
print("index: ", ages.index)

### Question 2

Label vs position. -> both print `24`; `plain[0]` prints `10`.

For `ages`, the two lookups are unmistakably different operations that
happen to agree: `"Sara"` is a label, `1` is a position.

`plain` is the trap. Its labels are `0, 1, 2`, so `plain[0]` is a *label*
lookup that reads exactly like a position lookup. It returns `10` — the
first element — so you get the right answer for the wrong reason and
learn nothing. The day someone hands you a Series indexed `[3, 1, 2]`,
`plain[0]` raises and the habit breaks. Use `.iloc` when you mean
position and `.loc` when you mean label; worksheet 04 is entirely about
this.

In [ ]:
print("by label:   ", ages["Sara"])
print("by position:", ages.iloc[1])

# plain's labels ARE 0,1,2 -- so plain[0] is a LABEL lookup that happens to
# look like a position. With `ages` there is no such confusion, because no
# label is an integer.
print("plain[0]:   ", plain[0])

### Question 3

`ages.mean()` -> `22.333333333333332`. The slide shows `22.33`.

67 divided by 3 has no exact binary representation, so what is stored is
slightly off and Python prints all seventeen significant digits it has.
`round(ages.mean(), 2)` gives `22.33` — so the slide is showing the
*rounded* value without saying so.

`ages.mean() == 22.33` is **False**. That is the part that matters. If you
write a test or a filter that compares a computed mean to a literal like
`22.33`, it fails, and it fails silently in the sense that nothing raises —
you just get the branch you did not expect. Compare with a tolerance, or
round both sides, but never compare a computed float for exact equality.

In [ ]:
print("ages.mean():      ", ages.mean())
print("slide shows:       22.33")
print("round(...,2):     ", round(ages.mean(), 2))

# 67 / 3 has no exact binary representation, so the stored value is very
# slightly off and Python prints every digit it has.
print("is it exactly 22.33?", ages.mean() == 22.33)

### Question 4

`max` 24, `min` 21, `sum` 67, `len` 3. -> by hand `22.333333333333332`, matching `.mean()` exactly.

`sum() / len()` gives bit-for-bit the same float as `.mean()` here, so the
comparison is `True`. Do not read too much into that — it holds for three
small integers. On a large float column, `.mean()` and a hand-rolled
`sum()/len()` can disagree in the last digits, because they do not
necessarily add the numbers in the same order. Worksheet 12 shows exactly
that happening on the real sales file.

In [ ]:
print("max:", ages.max())
print("min:", ages.min())
print("sum:", ages.sum())
print("len:", len(ages))

by_hand = ages.sum() / len(ages)
print("by hand:", by_hand)
print("matches .mean():", by_hand == ages.mean())

PART B — comparisons produce a Series, not a single answer

### Question 5

`ages > 21` -> a Series of three booleans (`True`, `True`, `False`), `dtype: bool`. `.sum()` -> `2`.

The comparison did not collapse to one answer — it was applied to every
element and kept the labels. That returned object is the thing you hand
back to the Series to filter it, which is Q6.

Summing booleans counts them because `True` is 1 and `False` is 0. It
reads oddly the first time and then becomes the most natural way to count
matching rows in the whole library.

In [ ]:
mask = ages > 21
print(mask)
print()
print("dtype:", mask.dtype)

# True is 1 and False is 0, so summing a boolean Series counts the Trues.
print("how many over 21:", mask.sum())

### Question 6

`ages[ages > 21]` -> Abdullah 22 and Sara 24, 2 rows, matching the `.sum()` count.

Ahmed is gone because his entry in the mask was `False`. Nothing was
re-sorted and nothing was renumbered — the two surviving rows kept their
own labels, which is why you can still tell who they are.

That is the difference between this and filtering a list: a filtered list
loses track of which positions survived, so you need a second structure to
remember the names. Here the names came along for free.

In [ ]:
over = ages[ages > 21]
print(over)
print()
print("rows kept:", len(over))
print("matches the .sum() count:", len(over) == (ages > 21).sum())

### Question 7

`> 21` keeps 2; `>= 21` keeps 3. -> Ahmed is the only difference.

Ahmed is exactly 21, so he sits on the boundary and the operator decides
him alone. 'Over 21' is `>`; 'at least 21' or '21 and over' is `>=`.

English is genuinely ambiguous here — 'students 21 and over' and 'students
over 21' differ by one person and most people read them as the same
sentence. When a stakeholder asks for a threshold, the useful question is
not 'what number' but 'is the number itself in or out'.

In [ ]:
print("ages > 21 count: ", (ages > 21).sum())
print("ages >= 21 count:", (ages >= 21).sum())
print()
print(ages >= 21)

# "over 21"     -> strict >,  excludes Ahmed
# "at least 21" -> >=,        includes Ahmed
# One row of difference, and the row is a person.

PART C — the index is doing work you did not ask for

### Question 8

`ages + bonus` -> Abdullah 23, Ahmed 22, Sara 25 — **and the rows come back in alphabetical order**.

`bonus` was written in reverse order and it made no difference to the
arithmetic: Abdullah is 22 and got 23, not 25. Pandas matched on the
label, not the position. Written as two Python lists, `[22,24,21]` and
`[1,1,1]` reversed, you would have added Abdullah's age to Ahmed's bonus
and never noticed.

The second, quieter thing: the result is ordered `Abdullah, Ahmed, Sara` —
alphabetical — not the `Abdullah, Sara, Ahmed` you started with. When the
two indexes are not in the same order, Pandas builds the union and sorts
it. Your row order is not preserved through arithmetic, so never rely on
position after an operation like this.

In [ ]:
bonus = pd.Series([1, 1, 1], index=["Ahmed", "Sara", "Abdullah"])
print("ages:")
print(ages)
print()
print("bonus (reversed order):")
print(bonus)
print()
print("ages + bonus:")
print(ages + bonus)

# Abdullah is 22 and gets 23, not 25. Pandas matched on the LABEL, so the
# order the two Series were written in never mattered.

### Question 9

`ages + partial` -> Abdullah `NaN`, Ahmed `NaN`, Nadia `NaN`, Sara `25.0`, `dtype: float64`.

Three separate things went wrong quietly and none of them raised:

1. **Nadia exists now.** She was never in `ages`. The result index is the
   *union* of the two, so a label present in only one operand still gets a
   row — filled with `NaN`.
2. **Abdullah and Ahmed became `NaN`.** They have ages; they just have no
   bonus. `22 + nothing` is not 22, it is unknown.
3. **`int64` became `float64`.** `NaN` is a float, so the column has to
   widen to hold it. Your ages are no longer integers, and `22` will print
   as `22.0` from here on.

If you wanted the missing side treated as zero, `ages.add(partial,
fill_value=0)` says so explicitly. The plain `+` deliberately refuses to
guess.

In [ ]:
partial = pd.Series([1, 1], index=["Sara", "Nadia"])
print(ages + partial)
print()
print("dtype:", (ages + partial).dtype)

# Two things happened silently:
#  1. Nadia appeared, with NaN, because the result index is the UNION.
#  2. int64 became float64, because NaN is a float and the column has to
#     hold it. Your integer ages are no longer integers.

### Question 10

`ages["Nadia"]` -> **raises** `KeyError: 'Nadia'`.

Put this next to Q9 and the pair is the lesson. The *same* missing label:

- **added** to the Series -> a silent `NaN` row and no complaint;
- **asked for** directly -> an immediate `KeyError`.

Pandas is strict when you name one thing and permissive when you combine
whole objects. That asymmetry is not a quirk to memorise, it is the
library's whole posture: a lookup is a question with one right answer, so
a miss is an error; an alignment is a merge of two label sets, so a miss
is just an absence.

The practical consequence is that bugs from Q9 reach production and bugs
from Q10 do not.

In [ ]:
print(ages["Nadia"])